# 0.0 Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sklearn.metrics as mt
from xgboost import XGBRegressor

# 0.1 - Load Datasets

In [2]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [3]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [4]:
# Instanciar o modelo com parâmetros default
model_def = XGBRegressor(random_state=42)

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [5]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

--- Performance no Treino (Default) ---
R²:   0.7736
MSE:  108.24
RMSE: 10.40
MAE:  7.68
MAPE: 347.80%


## Passo 3 — Performance na validação (default)

In [6]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

--- Performance na Validação (Default) ---
R²:   0.2235
MSE:  370.80
RMSE: 19.26
MAE:  14.62
MAPE: 730.89%


## Passo 4 — Ajuste de hiperparâmetros

In [7]:
print("--- Testando Múltiplos Hiperparâmetros no XGBoost (Regressão) ---")
melhor_n_estimadores = 100
melhor_max_depth = 3
melhor_learning_rate = 0.1
melhor_r2_val = -999

# Listas de hiperparâmetros para o cruzamento no loop
list_n_estimators = [100, 200, 500]
list_max_depth = [3, 5, 7]
list_learning_rate = [0.01, 0.05, 0.1]

# LOOP TRIPLO: Cruza N_Estimators x Max_Depth x Learning_Rate
for n_est in list_n_estimators:
    for md in list_max_depth:
        for lr in list_learning_rate:
            
            # Instancia o regressor do XGBoost
            model = XGBRegressor(
                n_estimators=n_est,
                max_depth=md,
                learning_rate=lr,
                n_jobs=-1,
                random_state=42
            )
            
            try:
                model.fit(X_train, y_train.values.ravel())
                
                yhat_train = model.predict(X_train)
                yhat_val = model.predict(X_val)
                
                r2_train = mt.r2_score(y_train, yhat_train)
                r2_val = mt.r2_score(y_val, yhat_val)
                rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
                
                print(f"N_Est: {n_est} | Max_Depth: {md} | LR: {lr:<4} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f}")
                
                if r2_val > melhor_r2_val:
                    melhor_r2_val = r2_val
                    melhor_n_estimators = n_est
                    melhor_max_depth = md
                    melhor_learning_rate = lr
                    
            except Exception as e:
                print(f"Erro na combinação N_Est {n_est} | Max_Depth {md} | LR {lr}: {e}")

print("=" * 85)
print(f"-> VENCEDOR DO ENSAIO XGBOOST:")
print(f"Melhor N_Estimators: {melhor_n_estimators}")
print(f"Melhor Max_Depth: {melhor_max_depth}")
print(f"Melhor Learning_Rate: {melhor_learning_rate}")
print(f"Maior R² obtido na Validação: {melhor_r2_val:.4f}\n")

--- Testando Múltiplos Hiperparâmetros no XGBoost (Regressão) ---
N_Est: 100 | Max_Depth: 3 | LR: 0.01 | R² Treino: 0.0669 | R² Val: 0.0593 | RMSE Val: 21.19


N_Est: 100 | Max_Depth: 3 | LR: 0.05 | R² Treino: 0.1404 | R² Val: 0.1035 | RMSE Val: 20.69
N_Est: 100 | Max_Depth: 3 | LR: 0.1  | R² Treino: 0.1797 | R² Val: 0.1183 | RMSE Val: 20.52


N_Est: 100 | Max_Depth: 5 | LR: 0.01 | R² Treino: 0.1241 | R² Val: 0.0893 | RMSE Val: 20.85


N_Est: 100 | Max_Depth: 5 | LR: 0.05 | R² Treino: 0.2658 | R² Val: 0.1453 | RMSE Val: 20.20


N_Est: 100 | Max_Depth: 5 | LR: 0.1  | R² Treino: 0.3672 | R² Val: 0.1767 | RMSE Val: 19.83


...

N_Est: 500 | Max_Depth: 5 | LR: 0.05 | R² Treino: 0.5595 | R² Val: 0.2183 | RMSE Val: 19.32


N_Est: 500 | Max_Depth: 5 | LR: 0.1  | R² Treino: 0.7420 | R² Val: 0.2444 | RMSE Val: 19.00


N_Est: 500 | Max_Depth: 7 | LR: 0.01 | R² Treino: 0.4795 | R² Val: 0.2080 | RMSE Val: 19.45


N_Est: 500 | Max_Depth: 7 | LR: 0.05 | R² Treino: 0.8353 | R² Val: 0.2786 | RMSE Val: 18.56


N_Est: 500 | Max_Depth: 7 | LR: 0.1  | R² Treino: 0.9559 | R² Val: 0.2876 | RMSE Val: 18.44
-> VENCEDOR DO ENSAIO XGBOOST:
Melhor N_Estimators: 500
Melhor Max_Depth: 7
Melhor Learning_Rate: 0.1
Maior R² obtido na Validação: 0.2876



## Passo 5 — Performance com modelo tunado (treino e validação)

In [8]:
# Modelo com os melhores hiperparâmetros encontrados, treinado apenas em X_train
model_tunado = XGBRegressor(
    n_estimators=melhor_n_estimators,
    max_depth=melhor_max_depth,
    learning_rate=melhor_learning_rate,
    n_jobs=-1,
    random_state=42
)
model_tunado.fit(X_train, y_train.values.ravel())

yhat_train_tunado = model_tunado.predict(X_train)
yhat_val_tunado   = model_tunado.predict(X_val)

r2_train_tunado   = mt.r2_score(y_train, yhat_train_tunado)
mse_train_tunado  = mt.mean_squared_error(y_train, yhat_train_tunado)
rmse_train_tunado = np.sqrt(mse_train_tunado)
mae_train_tunado  = mt.mean_absolute_error(y_train, yhat_train_tunado)
mape_train_tunado = mt.mean_absolute_percentage_error(y_train, yhat_train_tunado)

r2_val_tunado   = mt.r2_score(y_val, yhat_val_tunado)
mse_val_tunado  = mt.mean_squared_error(y_val, yhat_val_tunado)
rmse_val_tunado = np.sqrt(mse_val_tunado)
mae_val_tunado  = mt.mean_absolute_error(y_val, yhat_val_tunado)
mape_val_tunado = mt.mean_absolute_percentage_error(y_val, yhat_val_tunado)

print("--- Performance com Modelo Tunado ---")
print(f"Treino    → R²: {r2_train_tunado:.4f} | RMSE: {rmse_train_tunado:.2f} | MAE: {mae_train_tunado:.2f} | MAPE: {mape_train_tunado*100:.2f}%")
print(f"Validação → R²: {r2_val_tunado:.4f} | RMSE: {rmse_val_tunado:.2f} | MAE: {mae_val_tunado:.2f} | MAPE: {mape_val_tunado*100:.2f}%")

--- Performance com Modelo Tunado ---
Treino    → R²: 0.9559 | RMSE: 4.59 | MAE: 2.96 | MAPE: 116.65%
Validação → R²: 0.2876 | RMSE: 18.44 | MAE: 13.17 | MAPE: 673.04%


## Passo 6 — União treino + validação

In [9]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

X_train_final shape: (15068, 13)
y_train_final shape: (15068, 1)


## Passo 7 — Retreino com melhores parâmetros

In [10]:
# Retreino com os melhores hiperparâmetros sobre o conjunto final (treino + validação)
model_final = XGBRegressor(
    n_estimators=melhor_n_estimators,
    max_depth=melhor_max_depth,
    learning_rate=melhor_learning_rate,
    n_jobs=-1,
    random_state=42
)
model_final.fit(X_train_final, y_train_final.values.ravel())

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## Passo 8 — Performance no teste

In [11]:
# Predição no conjunto de teste (única vez que X_test é tocado)
yhat_test = model_final.predict(X_test)

# Métricas no conjunto de TESTE
r2_test   = mt.r2_score(y_test, yhat_test)
mse_test  = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

print("--- Performance no Teste (Final) ---")
print(f"R²:   {r2_test:.4f}")
print(f"MSE:  {mse_test:.2f}")
print(f"RMSE: {rmse_test:.2f}")
print(f"MAE:  {mae_test:.2f}")
print(f"MAPE: {mape_test * 100:.2f}%")

--- Performance no Teste (Final) ---
R²:   0.3678
MSE:  307.81
RMSE: 17.54
MAE:  12.68
MAPE: 616.91%


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [12]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   r2_train_tunado,   r2_val_tunado,   r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, rmse_train_tunado, rmse_val_tunado, rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  mae_train_tunado,  mae_val_tunado,  mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%', f'{mape_val_def*100:.2f}%',
              f'{mape_train_tunado*100:.2f}%', f'{mape_val_tunado*100:.2f}%',
              f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))


--- Quadro Comparativo — Diagnóstico Completo ---
           Conjunto       R²      RMSE       MAE    MAPE
   Treino (Default) 0.773570 10.403668  7.681679 347.80%
Validação (Default) 0.223483 19.256065 14.616771 730.89%
    Treino (Tunado) 0.955905  4.591049  2.960484 116.65%
 Validação (Tunado) 0.287561 18.444451 13.165884 673.04%
      Teste (Final) 0.367819 17.544505 12.678854 616.91%
